---
# Методы статистической обработки информации
---
## Лабораторная работа 1 / Корреляционный анализ / Вариант 15
---

---
## Подготовка
---

### Импорт библиотек


In [2]:
from math             import log, e
from MoSDP_data       import get_data
from MoSDP_operations import complement, det

### Получение данных варианта

In [3]:
variant  = int(input("Введите номер варианта: "))

var_data = get_data(variant) # матрица исходных данных размерности k*n
k        = len(var_data)     # количество признаков
n        = len(var_data[0])  # количество предприятий

---
## Задание 1
---

### Средние арифметические отклонения

In [4]:
X_means = []
for x in var_data:
    X_means.append(sum(x) / len(x))

print("X:", X_means)

X: [7.970377358490566, 0.7354716981132075, 0.3011320754716981, 1.5260377358490564, 5.969056603773585, 19.570377358490568]


### Среднеквадратические отклонения


In [5]:
S = []
for j in range(k):
    s = 0
    for i in var_data[j]:
        s += (i - X_means[j]) ** 2
    S.append((s / n) ** 0.5)

print("S:", S)

S: [2.5854826652517997, 0.05272091124043734, 0.15372924467802906, 0.40387434820998896, 2.3608906731106623, 4.657878277314663]


### Матрица парных коэффициентов корреляции

In [6]:
R = [[[] for i in range(k)] for j in range(k)]
for l in range(len(R)):
    for j in range(l, len(R)):
        if j == l:
            R[l][j] = 1
        else:
            s = 0
            for i in range(n):
                s += (var_data[j][i] - X_means[j]) * (var_data[l][i] - X_means[l])
            R[l][j] = R[j][l] = (s / n) / (S[j] * S[l])

for r in R:
    print(f"[{"| ".join(f"{i:<{25}}" for i in r)}]")

[1                        | 0.054840792487636555     | 0.15115051013848507      | 0.0003555860122828476    | 0.23405035667830304      | 0.018075047394182264     ]
[0.054840792487636555     | 1                        | -0.06874216425985696     | -0.016615678293707227    | 0.2193289370025393       | -0.9402329788072865      ]
[0.15115051013848507      | -0.06874216425985696     | 1                        | 0.4333027759567459       | -0.04467957562887469     | 0.14675271575071883      ]
[0.0003555860122828476    | -0.016615678293707227    | 0.4333027759567459       | 1                        | -0.5084487611839539      | 0.03736158062663443      ]
[0.23405035667830304      | 0.2193289370025393       | -0.04467957562887469     | -0.5084487611839539      | 1                        | -0.14710495223940964     ]
[0.018075047394182264     | -0.9402329788072865      | 0.14675271575071883      | 0.03736158062663443      | -0.14710495223940964     | 1                        ]


---
## Задание 2
---

### Матрица частных коэффициентов корреляции

In [7]:
r_partial = [[[] for i in range(k)] for j in range(k)]
for i in range(len(r_partial)):
    for j in range(i, len(r_partial)):
        if i == j:
            r_partial[i][j] = 1
        else:
            r_partial[i][j] = r_partial[j][i] = - (complement(R, i, j)) / ((complement(R, i, i) * complement(R, j, j)) ** 0.5)

for r in r_partial:
    print(f"[{"| ".join(f"{i:<{25}}" for i in r)}]")

[1                        | 0.124446434986928        | 0.08693689662057558      | 0.06227701352246065      | 0.20782087145215675      | 0.13108308522239173      ]
[0.124446434986928        | 1                        | 0.12161905587924753      | 0.11454931336658208      | 0.24130787660988803      | -0.9437972750086439      ]
[0.08693689662057558      | 0.12161905587924753      | 1                        | 0.4562097875551488       | 0.18121101563658776      | 0.1738446723512375       ]
[0.06227701352246065      | 0.11454931336658208      | 0.4562097875551488       | 1                        | -0.5614261812968557      | 0.06506576460114392      ]
[0.20782087145215675      | 0.24130787660988803      | 0.18121101563658776      | -0.5614261812968557      | 1                        | 0.1659083526590121       ]
[0.13108308522239173      | -0.9437972750086439      | 0.1738446723512375       | 0.06506576460114392      | 0.1659083526590121       | 1                        ]


### Значимость

In [8]:
r_critical = 0.288 # (По "таблице 5" при: a = 0.05 и v = n - l - 2 = 53 - 4 - 2 = 47)

for i in range(len(r_partial)):
    for j in range(i + 1, len(r_partial)):
        res = (' <=', 'не')
        if abs(r_partial[i][j]) > r_critical:
            res = (' > ', '  ')

        print(f"r_{i + 1}{j + 1}/ост.: |{r_partial[i][j]:{6}.3f}| {res[0]} {r_critical}: гипотеза H0 {res[1]} отвергается")

r_12/ост.: | 0.124|  <= 0.288: гипотеза H0 не отвергается
r_13/ост.: | 0.087|  <= 0.288: гипотеза H0 не отвергается
r_14/ост.: | 0.062|  <= 0.288: гипотеза H0 не отвергается
r_15/ост.: | 0.208|  <= 0.288: гипотеза H0 не отвергается
r_16/ост.: | 0.131|  <= 0.288: гипотеза H0 не отвергается
r_23/ост.: | 0.122|  <= 0.288: гипотеза H0 не отвергается
r_24/ост.: | 0.115|  <= 0.288: гипотеза H0 не отвергается
r_25/ост.: | 0.241|  <= 0.288: гипотеза H0 не отвергается
r_26/ост.: |-0.944|  >  0.288: гипотеза H0    отвергается
r_34/ост.: | 0.456|  >  0.288: гипотеза H0    отвергается
r_35/ост.: | 0.181|  <= 0.288: гипотеза H0 не отвергается
r_36/ост.: | 0.174|  <= 0.288: гипотеза H0 не отвергается
r_45/ост.: |-0.561|  >  0.288: гипотеза H0    отвергается
r_46/ост.: | 0.065|  <= 0.288: гипотеза H0 не отвергается
r_56/ост.: | 0.166|  <= 0.288: гипотеза H0 не отвергается


### Интервальные оценки

In [9]:
t = 1.96 # (По "таблице 1": Ф(t) = 0.95)
l = k - 2

for i in range(len(r_partial)):
    for j in range(i + 1, len(r_partial)):
        Z = 0.5 * (log(1 + r_partial[i][j]) - log(1 - r_partial[i][j])) # (По "таблице 6")

        Z1 = Z - t * ((1 / (n - l - 3)) ** 0.5)
        Z2 = Z + t * ((1 / (n - l - 3)) ** 0.5)
        Z_min, Z_max = min(Z1, Z2), max(Z1, Z2)

        r_min = (e ** (2 * Z_min) - 1) / (e ** (2 * Z_min) + 1)
        r_max = (e ** (2 * Z_max) - 1) / (e ** (2 * Z_max) + 1)

        if r_min < 0 < r_max:
            res = 'гипотеза H0 не отвергается'
        else:
            res = 'гипотеза H0    отвергается'

        print(f"r_{i + 1}{j + 1}/ост.: {r_min:{6}.3f} <= р <= {r_max:{6}.3f}: {res}")


r_12/ост.: -0.162 <= р <=  0.392: гипотеза H0 не отвергается
r_13/ост.: -0.199 <= р <=  0.359: гипотеза H0 не отвергается
r_14/ост.: -0.223 <= р <=  0.338: гипотеза H0 не отвергается
r_15/ост.: -0.078 <= р <=  0.462: гипотеза H0 не отвергается
r_16/ост.: -0.156 <= р <=  0.398: гипотеза H0 не отвергается
r_23/ост.: -0.165 <= р <=  0.389: гипотеза H0 не отвергается
r_24/ост.: -0.172 <= р <=  0.383: гипотеза H0 не отвергается
r_25/ост.: -0.043 <= р <=  0.489: гипотеза H0 не отвергается
r_26/ост.: -0.968 <= р <= -0.902: гипотеза H0    отвергается
r_34/ост.:  0.201 <= р <=  0.654: гипотеза H0    отвергается
r_35/ост.: -0.105 <= р <=  0.440: гипотеза H0 не отвергается
r_36/ост.: -0.113 <= р <=  0.434: гипотеза H0 не отвергается
r_45/ост.: -0.728 <= р <= -0.333: гипотеза H0    отвергается
r_46/ост.: -0.220 <= р <=  0.340: гипотеза H0 не отвергается
r_56/ост.: -0.121 <= р <=  0.427: гипотеза H0 не отвергается


## Задание 3

### Множественные коэффициенты корреляции (детерминации)

In [10]:
R_multiple = []
for i in range(k):
    R_multiple.append((1 - (det(R) / complement(R, i, i))) ** 0.5)

D = [i ** 2 for i in R_multiple]

print(f"Коэффициенты корреляции:   [{"| ".join(f"{i:<{20}}" for i in R_multiple)}]")
print(f"Коэффициенты детерминации: [{"| ".join(f"{i:<{20}}" for i in D)}]")

Коэффициенты корреляции:   [0.3186590586278281  | 0.9482079283799225  | 0.5241659322704921  | 0.6690381098079028  | 0.63245280597947    | 0.9471917262754672  ]
Коэффициенты детерминации: [0.1015435956455736  | 0.8990982754425443  | 0.27474992455299413 | 0.4476119923753314  | 0.39999655179130506 | 0.8971721663246996  ]


### Значимость

In [11]:
F_critical = 2.37 # (По "таблице 4" при: a = 0.05; v1 = k - 1 = 6 - 1 = 5 и v2 = n - k = 53 - 6 = 47)

for i in range(len(D)):
    res = (' <=', 'не')

    d = D[i]
    F = (d / (k - 1)) / ((1 - d) / (n - k))
    if F > F_critical:
        res = (' > ', '  ')

    print(f"r_{i + 1}/ост. = {F:{7}.3f} {res[0]} {F_critical}: гипотеза H0 {res[1]} отвергается")

r_1/ост. =   1.062  <= 2.37: гипотеза H0 не отвергается
r_2/ост. =  83.760  >  2.37: гипотеза H0    отвергается
r_3/ост. =   3.561  >  2.37: гипотеза H0    отвергается
r_4/ост. =   7.617  >  2.37: гипотеза H0    отвергается
r_5/ост. =   6.267  >  2.37: гипотеза H0    отвергается
r_6/ост. =  82.015  >  2.37: гипотеза H0    отвергается
